# 🐕 Dogebone 02: a trained model of DOGE's direction after Musk's DOGE posts

**Question.** On a day Elon Musk posts about DOGE, will DOGE close **higher or lower** *h* days later (h = 1, 3, 7)? How sure is the model (**conviction**)?

Notebook 01 scored posts with a hand-built keyword lexicon and checked the calls. This notebook **trains a model** on Musk's 2021 DOGE posts
plus the market state, then **tests it on data it never saw**:

| Test | Trained on | Tested on |
|---|---|---|
| **Held-out months** (pre-registered) | Post days in Jan–Aug 2021 whose outcome was known by Aug 31 | Every post day in Sep–Dec 2021 |
| **Walk-forward** | All earlier post days, refit at the start of every month | Each month from Mar to Dec 2021 |

| Section | What it does |
|---|---|
| 1–2 | Setup, then one row per **post day** with text and market features (all known at the anchor close) |
| 3 | The model: L2 logistic regression, fit on Jan–Aug |
| 4 | **Test 1:** held-out Sep–Dec, post day by post day |
| 5 | **Test 2:** walk-forward, vs. no-skill, majority-class and momentum baselines |
| 6 | Is it the words or the market? Feature ablation and permutation tests |
| 7 | Does conviction mean anything? |
| 8 | Final model on all of 2021 → `models/*.json` |
| 9 | **Use it:** score new posts with `Predictor` |
| 10 | Export charts and `presentation/model_findings.md` |

**Rules we committed to before testing:** the feature list, the penalty (`L2 = 5`), the horizons, and the focus horizon (1 day) are fixed.
Nothing is tuned on the test months. All three horizons are reported, not just the best one.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dogebone").exists())
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from dogebone import Config, model as M, pipeline, plots, report

pd.set_option("display.max_columns", 40, "display.width", 220, "display.max_colwidth", 80)
plots.use_style()

cfg = Config(analysis_start="2021-01-01", analysis_end="2021-12-31", train_end="2021-08-31", horizons=(1, 3, 7))
FOCUS_H = 1             # headline horizon, fixed before testing (same as notebook 01)
L2 = M.DEFAULT_L2       # ridge penalty, fixed a priori
FIRST_TEST = "2021-03-01"  # walk-forward: first month predicted (needs >= 15 earlier post days)
N_PERM = 300            # shuffles per permutation test
print("Repo root:", ROOT)

## 2. From posts to post days

Prices are daily, so every DOGE post sharing an anchor close becomes **one row** (a *post day*). Each feature is known at the anchor close
or at post time, whichever is later:

| Feature | Meaning |
|---|---|
| `lex_score` | Lexicon sentiment of the day's posts (notebook 01's daily score, −1 to +1) |
| `lex_any` | Did any post use a sentiment word? (only about a quarter of post days do) |
| `n_posts_log` | How many DOGE posts that day, as log(1 + n) |
| `text_share` | Share of posts that say "Doge" outright, vs. replies that only tag a DOGE account |
| `has_original` | At least one original post (shown to all followers), not only replies |
| `gap_days_log` | Days since the previous DOGE post day, as log(1 + d), capped at 30 |
| `mom_1d`, `mom_7d` | DOGE return over the 1 and 7 days into the anchor close |
| `vol_7d` | Standard deviation of daily log returns over the last 7 days |
| `dd_30d` | Distance below the 30-day high |

The target `y_hd` is 1 if DOGE closed higher *h* days after the anchor close, otherwise 0. Likes and reposts are **excluded**,
because they keep accumulating after the post and partly reflect the price reaction (look-ahead).

In [ ]:
prep = pipeline.prepare(cfg)
DATA_MODE = prep.data_mode
if DATA_MODE != "raw":
    display(Markdown("> ## ⚠️ Running on SYNTHETIC sample data. Don't present these results."))

events = M.build_event_features(prep.posts, prep.prices["close"], cfg.horizons, username="elonmusk")
events["period"] = np.where(events["anchor_date"] <= pd.Timestamp(cfg.train_end), "train (Jan–Aug)", "test (Sep–Dec)")
print(f"Data mode: {DATA_MODE}  |  {len(prep.posts)} posts -> {len(events)} post days")
display(events.groupby("period").agg(post_days=("anchor_date", "size"), posts=("n_posts", "sum"),
                                     **{f"up_rate_{h}d": (f"y_{h}d", "mean") for h in cfg.horizons}).style.format(
    {f"up_rate_{h}d": "{:.0%}" for h in cfg.horizons}))
display(events.groupby(events["anchor_date"].dt.to_period("M")).size().rename("post days").to_frame().T)

In [ ]:
display(events[M.ALL_FEATURES].describe().T[["mean", "std", "min", "50%", "max"]].style.format("{:.3f}"))
display(events[["anchor_date", "n_posts", *M.ALL_FEATURES, "fwd_ret_1d", "y_1d", "texts"]].head(5))

## 3. The model

**L2-regularized logistic regression** gives P(DOGE up over the next *h* days | features).

- Features are standardized with the **training** mean and SD, so coefficients read as *log-odds per 1 SD*.
- The penalty shrinks every coefficient toward 0 (the base rate), which matters with about 50 training days.
- **Call:** UP if P(up) ≥ 50%, otherwise DOWN.
- **Conviction:** |2·P(up) − 1|. LOW means 40–60%, MEDIUM 60–70% (or 30–40%), HIGH 70%+ (or 30% and below).

Training rows are **purged**: a post day is used for training only if its outcome close (anchor + *h* days) was already known before the
test period starts, so 7-day windows never straddle the train/test boundary.

In [ ]:
ho_models, ho_preds = {}, {}
for h in cfg.horizons:
    ho_models[h], ho_preds[h] = M.holdout(events, h, cfg.train_end, M.ALL_FEATURES, L2)
    print(f"{h}d model: trained on {ho_models[h].n_train_} post days ({ho_models[h].train_start_} to {ho_models[h].train_end_}), "
          f"training up-rate {ho_models[h].base_rate_:.0%}")
display(Markdown(f"**Coefficients of the {FOCUS_H}-day model trained on Jan–Aug** (log-odds per 1 SD)"))
display(ho_models[FOCUS_H].coefficients().style.format({"coef_per_sd": "{:+.3f}", "odds_ratio_per_sd": "{:.2f}",
                                                       "train_mean": "{:.3f}", "train_sd": "{:.3f}"}).hide(axis="index"))

## 4. Test 1: the held-out months (Sep–Dec 2021)

The models above saw **no data after Aug 31**. Each Sep–Dec post day is predicted exactly once.

| Metric | Meaning |
|---|---|
| `hit_rate` | Share of calls whose direction matched the move |
| `null_hit_rate` | Hit rate expected if the calls were independent of outcomes, given the UP/DOWN mix. **The bar to beat.** |
| `p_value` | One-sided binomial test of the hit rate against `null_hit_rate` |
| `majority_hit_rate` | Always calling whatever happened more often in training |
| `momentum_hit_rate` | "The last *h* days' direction continues" |
| `auc` | Probability that a random up-day got a higher P(up) than a random down-day (0.5 = no ranking skill) |
| `brier_skill` | 1 − Brier / Brier of always forecasting the training up-rate. Above 0 means the probabilities beat the base rate. |

In [ ]:
fmt = {"hit_rate": "{:.1%}", "null_hit_rate": "{:.1%}", "edge_pp": "{:+.1f}", "p_value": "{:.3f}", "majority_hit_rate": "{:.1%}",
       "momentum_hit_rate": "{:.1%}", "up_rate": "{:.0%}", "share_called_up": "{:.0%}", "auc": "{:.2f}", "brier": "{:.3f}",
       "brier_ref": "{:.3f}", "brier_skill": "{:+.3f}", "log_loss": "{:.3f}", "mean_ret_when_up": "{:+.2%}", "mean_ret_when_down": "{:+.2%}"}
holdout_scores = pd.DataFrame([M.score_predictions(ho_preds[h], h) for h in cfg.horizons])
display(holdout_scores.style.format(fmt).hide(axis="index"))

In [ ]:
# The test on the current data: every held-out post day, what the model said, and what DOGE did.
t = ho_preds[FOCUS_H].copy()
t["first post"] = t["texts"].str.split(" | ", regex=False).str[0].str.slice(0, 70)
display(t[["anchor_date", "n_posts", "first post", "p_up", "direction", "conviction_bucket", f"fwd_ret_{FOCUS_H}d", "actual", "hit"]]
        .style.format({"p_up": "{:.0%}", f"fwd_ret_{FOCUS_H}d": "{:+.2%}"}).hide(axis="index"))
test_start = (pd.Timestamp(cfg.train_end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
fig_calls = plots.model_calls_timeline(prep.returns, ho_preds[FOCUS_H], test_start, cfg.analysis_end,
                                       title=f"Held-out test: {FOCUS_H}-day calls on every Sep–Dec 2021 Musk DOGE post day")
plt.show()

## 5. Test 2: walk-forward (refit every month)

A single four-month holdout is only about 20 post days. The walk-forward test uses more of the year without ever peeking ahead.
At the start of each month from March, the model is refit on **all earlier post days whose outcome is already known**, then predicts that month's post days.

In [ ]:
wf_preds = {h: M.walk_forward(events, h, M.ALL_FEATURES, L2, first_test=FIRST_TEST) for h in cfg.horizons}
wf_scores = pd.DataFrame([M.score_predictions(wf_preds[h], h) for h in cfg.horizons])
display(wf_scores.style.format(fmt).hide(axis="index"))
fig_base = plots.model_vs_baselines(wf_scores, title="Walk-forward: model hit rate vs. simple rules (Mar–Dec 2021)")
plt.show()

monthly = pd.concat([wf_preds[h].assign(horizon_d=h) for h in cfg.horizons])
display(Markdown("**Hit rate by month predicted** (n in brackets)"))
display(monthly.groupby(["fold", "horizon_d"])["hit"].agg(lambda s: f"{s.mean():.0%} ({s.notna().sum()})").unstack())

## 6. Is it the words, or the market?

The same two tests with **text-only** and **market-only** feature sets show which half carries what skill there is.
Two permutation tests (walk-forward, all features) ask:

- **Outcomes shuffled:** is *anything* in the features informative?
- **Text shuffled, market kept:** do Musk's words add anything beyond the market state?

In [ ]:
wf_grid = M.evaluate_grid(events, cfg.horizons, scheme="walk_forward", l2=L2, first_test=FIRST_TEST)
ho_grid = M.evaluate_grid(events, cfg.horizons, scheme="holdout", train_end=cfg.train_end, l2=L2)
cols = ["features", "horizon_d", "n", "hit_rate", "null_hit_rate", "p_value", "auc", "brier_skill"]
display(Markdown("**Walk-forward**"), wf_grid[cols].style.format(fmt).hide(axis="index"))
display(Markdown("**Held-out Sep–Dec**"), ho_grid[cols].style.format(fmt).hide(axis="index"))

In [ ]:
perm = pd.DataFrame([M.permutation_test(events, h, M.ALL_FEATURES, L2, n_perm=N_PERM, seed=h, shuffle=s, first_test=FIRST_TEST)
                     for s in ("outcome", "text") for h in cfg.horizons])
display(perm.style.format({c: "{:.3f}" for c in perm.columns if perm[c].dtype.kind == "f"}).hide(axis="index"))

## 7. Does conviction mean anything?

If the probabilities are informative, HIGH-conviction calls should hit more often than LOW ones, and the realized up-rate should track the mean P(up).

In [ ]:
conv_tables = {h: M.by_conviction(wf_preds[h], h) for h in cfg.horizons}
for h in cfg.horizons:
    display(Markdown(f"**{h}-day, walk-forward**"),
            conv_tables[h].style.format({"mean_p_up": "{:.0%}", "realized_up_rate": "{:.0%}", "hit_rate": "{:.0%}"}).hide(axis="index"))

## 8. Final model on all of 2021 → `models/`

The deployable model is refit on **every** 2021 post day with a known outcome. Its skill is the out-of-sample skill estimated above,
**not** its in-sample fit. Each horizon is saved as plain JSON (coefficients, training means and SDs, metrics), so it can be read without this repo.

In [ ]:
final = {}
model_dir = cfg.root / "models"
for h in cfg.horizons:
    meta = {
        "data_mode": DATA_MODE, "account": "elonmusk", "analysis_window": [cfg.analysis_start, cfg.analysis_end],
        "anchor_mode": cfg.anchor_mode,
        "walk_forward_test": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in M.score_predictions(wf_preds[h], h).items()},
        "heldout_test": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in M.score_predictions(ho_preds[h], h).items()},
        "heldout_window": [test_start, cfg.analysis_end], "walk_forward_first_month": FIRST_TEST,
    }
    final[h] = M.fit_final(events, h, M.ALL_FEATURES, L2, **meta)
    path = final[h].save(model_dir / f"musk_doge_direction_{h}d.json")
    print(f"saved {path.relative_to(cfg.root)}  (trained on {final[h].n_train_} post days)")
coef_final = final[FOCUS_H].coefficients()
display(coef_final.style.format({"coef_per_sd": "{:+.3f}", "odds_ratio_per_sd": "{:.2f}", "train_mean": "{:.3f}",
                                 "train_sd": "{:.3f}"}).hide(axis="index"))
fig_coef = plots.model_coefficients(coef_final, title=f"What the {FOCUS_H}-day model learned from Musk's 2021 DOGE post days")
plt.show()

## 9. Use it: score new posts

`Predictor` loads the saved models plus the repo's prices and lexicon. Give it the post text and its UTC time. It needs a DOGE close for the
post's anchor date (add newer prices to `data/raw/prices/` to score recent posts). When later prices exist, it also shows what actually happened.

First, an honest demo: Musk's last DOGE posts of 2021 scored by the **Jan–Aug model**, which had never seen them.

In [ ]:
oos = M.Predictor(ho_models, prep)
last_days = prep.posts["anchor_date"].drop_duplicates().nlargest(2)
last = prep.posts.loc[prep.posts["anchor_date"].isin(last_days), ["text", "created_at"]]
demo = oos.predict(last)
display(demo)
# Same model, same inputs -> must match the held-out predictions in section 4.
check = demo[demo["horizon_d"] == FOCUS_H].merge(ho_preds[FOCUS_H][["anchor_date", "p_up"]], on="anchor_date", suffixes=("", "_test"))
assert np.allclose(check["p_up"], check["p_up_test"], atol=1e-3), check

In [ ]:
# Any post, with the deployed (all-2021) models. Change the text and time and re-run.
pred = M.Predictor.from_repo(cfg=cfg)
pred.predict("Dogecoin is the people's crypto 🚀", posted_at="2021-12-20T15:00:00Z")

## 10. Export for the presentation

In [ ]:
fig_dir = cfg.presentation_dir / "figures"
mark = None if DATA_MODE == "raw" else "SYNTHETIC DATA"
figs = {"06_model_vs_baselines": fig_base, "07_model_heldout_calls": fig_calls, "08_model_coefficients": fig_coef}
saved = []
for name, fig in figs.items():
    plots.save(fig, name, cfg.figures_dir, fig_dir, watermark=mark)
    saved.append(f"presentation/figures/{name}.png")

cfg.processed_dir.mkdir(parents=True, exist_ok=True)
events.to_csv(cfg.processed_dir / "model_post_days.csv", index=False)
pd.concat([wf_preds[h].assign(horizon_d=h, test="walk_forward") for h in cfg.horizons]
          + [ho_preds[h].assign(horizon_d=h, test="holdout") for h in cfg.horizons]).to_csv(
    cfg.processed_dir / "model_predictions.csv", index=False)

findings = report.write_model_findings(
    cfg.presentation_dir / "model_findings.md", cfg, DATA_MODE, FOCUS_H, wf_grid, ho_grid, perm, conv_tables[FOCUS_H],
    coef_final, ho_preds[FOCUS_H], n_events=len(events), n_posts=len(prep.posts), figures=saved, l2=L2, first_test=FIRST_TEST,
)
display(Markdown(findings))

## 11. Caveats

1. **Small sample.** One account and one year give 69 post days, about 50 of them out-of-sample per horizon. A 10-point edge on 50 days is within noise.
2. **Several looks.** Three horizons × three feature sets × two tests. Judge the pre-registered focus horizon, and treat a lone p < 0.05 elsewhere as a lead, not a finding.
3. **Regime change.** Feb–May 2021 was a mania and Sep–Dec a slow bleed. Market features that meant "up" in spring can mean "down" in autumn, so the walk-forward and held-out numbers can disagree.
4. **Daily resolution and anchoring.** A post moves price within minutes; daily closes blur that. `nearest_close` lets at most 12h of pre-post movement in.
5. **Text the model can't see.** Image-only memes, and posts that move DOGE without saying "Doge", are invisible.